# YOLO — Cup Detector

Fine-tuning a pretrained YOLO model to detect cups in our own photos.

## 1. Sanity check

Run the original pretrained model on one photo to confirm everything works. It won't know "cup" as our class yet — this just proves the setup is fine.

In [ ]:
# sanity check: pretrained YOLO on one image
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model("images/image_1.jpeg")
results[0].show()

## 2. Build the dataset

Organize our labeled images into the folder layout YOLO expects (train/val split + `data.yaml`). Run this once.

In [ ]:
# build the YOLO dataset folder from the Label Studio export (run once)
import shutil, random
from pathlib import Path

random.seed(42)
IMG_SRC = Path("images_ready")
LBL_SRC = Path("labels/labels")
OUT = Path("dataset")
VAL_FRAC = 0.2

# pair each label with its image: "8d51379b-image_1" -> "image_1"
pairs = []
for lbl in LBL_SRC.glob("*.txt"):
    stem = lbl.stem.split("-", 1)[1]
    img = IMG_SRC / f"{stem}.png"
    if img.exists():
        pairs.append((img, lbl))
print(f"{len(pairs)} image/label pairs")

random.shuffle(pairs)
n_val = round(len(pairs) * VAL_FRAC)
splits = {"val": pairs[:n_val], "train": pairs[n_val:]}

for split, items in splits.items():
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)
    for img, lbl in items:
        shutil.copy(img, OUT / "images" / split / img.name)
        shutil.copy(lbl, OUT / "labels" / split / f"{img.stem}.txt")
    print(f"{split}: {len(items)}")

# data.yaml
classes = [c.strip() for c in Path("labels/classes.txt").read_text().splitlines() if c.strip()]
yaml = f"path: {OUT.resolve()}\ntrain: images/train\nval: images/val\nnames:\n"
yaml += "".join(f"  {i}: {c}\n" for i, c in enumerate(classes))
(OUT / "data.yaml").write_text(yaml)
print(yaml)

## 3. Fine-tune

Train the model on our cups, starting from the pretrained weights. Bigger model (`n < s < m < l < x`) = more accurate but slower.

In [6]:
#fine-tune
model = YOLO("yolov8s.pt")             # bigger model: n < s < m < l < x (auto-downloads)
model.train(
    data="dataset/data.yaml",
    epochs=100,                        
    imgsz=256,                          # matches the 256x256 processed images
    device="mps",                       # Apple Silicon; use 0 on NVIDIA, "cpu" otherwise
)

Ultralytics 8.4.71 🚀 Python-3.13.14 torch-2.12.1 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x15662dfd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048,    

## 4. Evaluate

Measure quality (precision, recall, mAP) on the held-out val images and save annotated predictions to `runs/detect/`.

In [ ]:
# evaluate & predict on the val split
metrics = model.val()
model.predict("dataset/images/val", save=True)

## 5. Test on new images

Load the trained weights (`best.pt`) and try the model on photos it has never seen.

In [10]:
# test the trained model on new images
from ultralytics import YOLO

trained = YOLO("runs/detect/train-2/weights/best.pt")

results = trained.predict("test_images", conf=0.4, save=True)  # runs on every image in the folder
for r in results:
    r.show()


image 1/5 /Users/gabss/Codes/USP/Processamento_Digital_Imagem/ImageProcFinal/test_images/ex1.png: 256x256 4 Cups, 19.4ms
image 2/5 /Users/gabss/Codes/USP/Processamento_Digital_Imagem/ImageProcFinal/test_images/ex2.png: 256x256 1 Cup, 14.9ms
image 3/5 /Users/gabss/Codes/USP/Processamento_Digital_Imagem/ImageProcFinal/test_images/ex3.png: 256x256 2 Cups, 15.0ms
image 4/5 /Users/gabss/Codes/USP/Processamento_Digital_Imagem/ImageProcFinal/test_images/ex4.png: 256x256 1 Cup, 14.5ms
image 5/5 /Users/gabss/Codes/USP/Processamento_Digital_Imagem/ImageProcFinal/test_images/ex5.png: 256x256 4 Cups, 14.7ms
Speed: 0.3ms preprocess, 15.7ms inference, 0.3ms postprocess per image at shape (1, 3, 256, 256)
Results saved to /Users/gabss/Codes/USP/Processamento_Digital_Imagem/ImageProcFinal/runs/detect/predict-5
